In [1]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")      # non-GUI backend, can't crash the kernel
import matplotlib.pyplot as plt
%matplotlib inline

full_df = pd.read_csv(r"C:\Users\sarav\projects\vowel-analysis\data\phoible.csv")

C:\Users\sarav\AppData\Local\Temp\ipykernel_11932\3880440929.py:7: DtypeWarning: Columns (0: SpecificDialect, 1: Allophones, 2: Marginal, 3: tone) have mixed types. Specify dtype option on import or set low_memory=False.
  full_df = pd.read_csv(r"C:\Users\sarav\projects\vowel-analysis\data\phoible.csv")


In [2]:
total_rows = len(full_df)
total_invs = full_df["InventoryID"].nunique()
total_langs = full_df["Glottocode"].nunique()

In [3]:
print(f"Rows: {total_rows}")
print(f"Distinct inventories: {total_invs}")
print(f"Distinct languages: {total_langs}")

Rows: 105467
Distinct inventories: 3020
Distinct languages: 2184


In [4]:
full_df["ISO6393"].nunique()


2098

In [5]:
iso_col = "ISO6393" 
glotto_col = "Glottocode"

iso_counts = full_df.groupby(iso_col)[glotto_col].nunique()

multi_glotto_isos = iso_counts[iso_counts > 1].index[:3]

examples = (
    full_df[full_df[iso_col].isin(multi_glotto_isos)][[iso_col, glotto_col, 'LanguageName']]
    .drop_duplicates()
    .sort_values(by=iso_col)
)

print(examples)

      ISO6393 Glottocode        LanguageName
7650      aer   east2379            ARRERNTE
75556     aer   east2379   Arrernte, Central
97958     aer   mpar1238    Central Arrernte
98045     aer   east2379    Eastern Arrernte
97826     amx   east2380  Eastern Anmatyerre
97870     amx   west2442  Western Anmatyerre
67816     boa   bora1263                Bora
67857     boa   mira1254              Miraña


In [6]:
glotto_inv_counts = full_df.groupby("Glottocode")["InventoryID"].nunique()

multi_inv_glottocodes = glotto_inv_counts[glotto_inv_counts >= 2].index[:3]

examples = (
    full_df[full_df["Glottocode"].isin(multi_inv_glottocodes)][["Glottocode", "InventoryID", "LanguageName"]]
    .drop_duplicates()
    .sort_values(by="Glottocode")
)

print(examples)

      Glottocode  InventoryID LanguageName
21532   abid1235          649       abidji
55176   abid1235         1526       Abidji
8693    abip1241          235       ABIPON
69215   abip1241         1914       Abipon
88680   abkh1244         2468       Abkhaz
92639   abkh1244         2552       Abkhaz


In [7]:
max_ids = full_df.groupby("Glottocode")["InventoryID"].max()
single_inv_df = full_df[full_df["InventoryID"].isin(max_ids)].copy()

print("Original rows:", len(full_df))
print("Filtered rows:", len(single_inv_df))
print("Unique Glottocodes:", single_inv_df["Glottocode"].nunique())
print("Unique Inventories:", single_inv_df["InventoryID"].nunique())

Original rows: 105467
Filtered rows: 75370
Unique Glottocodes: 2184
Unique Inventories: 2184


In [8]:
multi = full_df.groupby("Glottocode")["InventoryID"].nunique().loc[lambda x: x > 1].index

for g in multi.to_series().sample(5, random_state=0):
    invs = {i: set(grp["Phoneme"]) for i, grp in full_df[full_df["Glottocode"] == g].groupby("InventoryID")}
    shared, union = set.intersection(*invs.values()), set.union(*invs.values())
    print(g, "sizes:", [len(s) for s in invs.values()], "shared:", len(shared), "union:", len(union))
    print("   differing:", list(union - shared)[:10])

seco1241 sizes: [18, 26] shared: 18 union: 26
   differing: ['ɨ̃', 'ẽ', 'n', 'ũ', 'ĩ', 'ã', 'õ', 't̠ʃ']
stan1293 sizes: [40, 39, 39, 44, 41, 45, 39, 44, 40] shared: 21 union: 95
   differing: ['kʰ', 'ɛ', 'ɑe', 'iɪ', 'e', 'ʔ', 'oʊ', 'ɑ', 'ɛʉ', 'eː']
lako1247 sizes: [36, 36] shared: 26 union: 46
   differing: ['t̪ʰ|tʰ', 'ɛ', 'z', 't', 'ʔ', 'b', 'n̪|n', 'e̞', 'sʼ', 'ə̆']
iris1253 sizes: [68, 69, 50, 49, 52] shared: 11 union: 139
   differing: ['kʰ', 'bˠ', 'ɾ̪ʲ', 'βʲ', 'l̪ʲ', 'ɡ̟', 'uːə', 'eː', 'ʝ', 'ə̯']
taga1270 sizes: [28, 23, 24, 26] shared: 10 union: 47
   differing: ['ɛ', 'd̪', 'aː', 'uː', 'n̪', 'e', 't', 'ʔ', 'ts', 'ʊ']


In [ ]:
inv_sizes = full_df.groupby(["Glottocode", "InventoryID"])["Phoneme"].count()
span = inv_sizes.groupby("Glottocode").agg(["min", "max", "count"])
multi = span[span["count"] > 1]
print("average size gap:", (multi["max"] - multi["min"]).mean())

average size gap: 7.497175141242938


: 

In [ ]:
sizes = full_df.groupby("InventoryID").agg(Source=("Source", "first"), size=("Phoneme", "count"))
print(sizes.groupby("Source")["size"].describe())
sizes.boxplot(column="size", by="Source", rot=45)

        count       mean        std   min   25%   50%    75%    max
Source                                                             
aa      203.0  39.724138   8.771734  22.0  34.0  38.0  44.00   82.0
ea      390.0  43.292308  14.319767  19.0  34.0  40.0  49.75  133.0
er      392.0  24.038265   4.836568  16.0  21.0  23.0  26.00   44.0
gm      460.0  41.917391  14.541718  18.0  33.0  39.0  47.00  161.0
ph      389.0  34.089974  11.452035  14.0  25.0  33.0  41.00   90.0
ra      100.0  42.610000   8.491106  21.0  36.0  42.0  48.00   62.0
saphon  355.0  25.495775   6.184714  11.0  21.0  25.0  29.00   51.0
spa     197.0  38.406091  12.957055  17.0  28.0  37.0  45.00   94.0
upsid   451.0  30.966741  11.554364  11.0  23.5  29.0  36.00  141.0
uz       83.0  44.686747  14.378955  21.0  34.5  41.0  55.50   74.0


<Axes: title={'center': 'size'}, xlabel='Source'>